# Practical 5 — POS Tagging & Chunking

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To assign part-of-speech (POS) tags to tokens, demonstrate how the same word can be tagged differently depending on sentence context, and use chunking to extract noun phrases from the review corpus.

## Theory

**POS tagging** assigns a grammatical category (noun, verb, adjective, etc.) to each token, based on both the word itself and its surrounding context. NLTK's tagger uses the Penn Treebank tagset — a few common tags:

| Tag | Meaning | Tag | Meaning |
|-----|---------|-----|---------|
| NN | singular noun | VB | base verb |
| NNS | plural noun | VBD | past-tense verb |
| JJ | adjective | VBG | gerund/present participle |
| RB | adverb | DT | determiner |
| PRP | pronoun | IN | preposition |

Practical 4 already surfaced why context matters here: a POS tagger given an isolated word list has far less to work with than one given a full sentence. This practical tests that directly with a genuinely ambiguous word.

**Chunking** (shallow parsing) groups tagged tokens into meaningful phrases without building a full syntactic parse tree. A simple grammar like `NP: {<DT>?<JJ>*<NN.*>+}` — an optional determiner, any number of adjectives, followed by one or more nouns — is enough to pull out noun phrases like "the acting" or "a great soundtrack." For review text specifically, the noun phrases mentioned are a rough signal of what aspects of the movie people are actually talking about.

## Algorithm

1. POS-tag a full sample review and inspect the tags.
2. Tag the word "watch" in two different constructed sentences where it plays a different grammatical role, and check whether the tagger correctly distinguishes them.
3. Define a noun-phrase chunking grammar and apply it to a tagged review, inspecting the resulting parse tree.
4. Extract noun phrases from that review as plain text.
5. Run chunking across the whole dataset and find the most frequently mentioned noun phrases.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
import nltk

for pkg in ["averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg)
    except Exception as e:
        print(f"Skipped {pkg}: {e}")

import preprocessing
import tokenizer
import pos_chunking as posc

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
df["tokens"] = df["review"].apply(lambda r: tokenizer.regex_word_tokenize(preprocessing.clean_text(r)))
print(f"Loaded {len(df)} reviews")


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Loaded 15 reviews


### Step 1 — POS-tag a full review

In [2]:
sample_tokens = df.loc[0, "tokens"]
tagged = posc.pos_tag_tokens(sample_tokens)
for word, tag in tagged:
    print(f"{word:<15} {tag}")


this            DT
movie           NN
was             VBD
absolutely      RB
fantastic       JJ
ive             JJ
never           RB
seen            VBN
anything        NN
like            IN
it              PRP
before          IN


### Step 2 — Does context change the tag for an ambiguous word?

"Watch" can be a verb ("I would watch this again") or a noun ("not worth a second watch"). Tag both sentences and check what the tagger actually assigns to "watch" in each.

In [3]:
sentence_a = "I would love to watch this movie again"
sentence_b = "This movie is not worth a second watch"

tokens_a = tokenizer.regex_word_tokenize(sentence_a.lower())
tokens_b = tokenizer.regex_word_tokenize(sentence_b.lower())

tagged_a = posc.pos_tag_tokens(tokens_a)
tagged_b = posc.pos_tag_tokens(tokens_b)

watch_tag_a = [tag for word, tag in tagged_a if word == "watch"]
watch_tag_b = [tag for word, tag in tagged_b if word == "watch"]

print(f"Sentence A tags: {tagged_a}")
print(f"'watch' tagged as: {watch_tag_a}")
print()
print(f"Sentence B tags: {tagged_b}")
print(f"'watch' tagged as: {watch_tag_b}")


Sentence A tags: [('i', 'NN'), ('would', 'MD'), ('love', 'VB'), ('to', 'TO'), ('watch', 'VB'), ('this', 'DT'), ('movie', 'NN'), ('again', 'RB')]
'watch' tagged as: ['VB']

Sentence B tags: [('this', 'DT'), ('movie', 'NN'), ('is', 'VBZ'), ('not', 'RB'), ('worth', 'JJ'), ('a', 'DT'), ('second', 'JJ'), ('watch', 'NN')]
'watch' tagged as: ['NN']


### Step 3 — Chunking: build the parse tree for a review

In [4]:
tree = posc.parse_tree(tagged)
print(tree)


(S
  (NP this/DT movie/NN)
  was/VBD
  absolutely/RB
  fantastic/JJ
  ive/JJ
  never/RB
  seen/VBN
  (NP anything/NN)
  like/IN
  it/PRP
  before/IN)


### Step 4 — Extract noun phrases from that review

In [11]:
noun_phrases = posc.extract_noun_phrases(tagged)
print(noun_phrases)


['this movie', 'anything']


### Step 5 — Most frequent noun phrases across the whole dataset

In [6]:
top_phrases = posc.top_noun_phrases(df["tokens"].tolist(), top_k=15)
for phrase, count in top_phrases:
    print(f"{phrase:<25} {count}")


film                      2
way                       2
i                         2
this movie                1
anything                  1
dont waste                1
a ticket i                1
a solid great visuals     1
the plot                  1
the acting                1
this time                 1
meh                       1
fine i guess nothing      1
special wouldnt watch     1
the directors             1


---
## Output

*Run every cell above top to bottom, then paste or describe your actual output here — the tagged review, whether "watch" was tagged differently in the two sentences, the parse tree, the noun phrases from the sample review, and the top-15 noun phrase list. Don't fill this in until you've actually run it.*


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Did the tagger actually assign "watch" a different tag in sentence A vs sentence B? If not, why do you think it failed to disambiguate — is this word genuinely harder to distinguish than "outdid" was in Practical 4?
- Look at the noun phrases extracted from the sample review — are they sensible phrases, or does the simple grammar miss/mangle anything?
- Looking at the top-15 noun phrases across the dataset — do they actually reflect what these reviews are about (acting, plot, soundtrack, etc.), or is the list dominated by less useful phrases?
- Would this noun-phrase extraction be a useful step in a real aspect-based sentiment pipeline for this data? What would you want to add or fix first?

*(Write 4-6 sentences here in your own words once you've run the notebook.)*


---
## Viva Prep — Practice Questions

1. **What does a POS tagger actually predict, and what does it use to make that prediction?**
   It predicts the grammatical category of each token (noun, verb, adjective, etc.), using both the word itself and its surrounding context — the same word can get different tags depending on the sentence it's in.

2. **What is chunking, and how is it different from full parsing?**
   Chunking (shallow parsing) groups tagged tokens into flat, non-recursive phrases (like noun phrases) using a simple grammar, without building a complete nested syntactic tree the way a full parser would.

3. **Explain the noun-phrase grammar `NP: {<DT>?<JJ>*<NN.*>+}` in your own words.**
   It matches an optional determiner, followed by zero or more adjectives, followed by one or more nouns of any noun subtype — e.g. "the great acting" (DT + JJ + NN).

4. **Why is noun-phrase extraction useful for review-style text specifically?**
   Because the noun phrases people use tend to correspond to the aspects of the product/movie they're actually discussing (e.g. "the soundtrack", "the plot") — useful groundwork for aspect-based sentiment analysis rather than just overall sentiment.

5. **What's a limitation of this simple regex-based chunking grammar?**
   It only catches flat noun phrases matching that exact pattern — it would miss more complex noun phrases with prepositional phrases attached (e.g. "the acting in the second half") or coordinated nouns, since the grammar has no rule for those structures.
